# 0.03 — The debugger

Lesson: [`lessons/0.03_debugger.md`](../lessons/0.03_debugger.md)

## Problem 1 — trace table, then debugger

1. **On paper**, fill a trace table (one row per loop iteration: `i`, `gap`, `biggest`) for `largest_gap([3, 11, 4, 4, 12])`.
2. Put a breakpoint on the `gap = ...` line and use **Debug Cell**. Step with `F10` and compare with your table.
3. Write the final answer your table predicted as a comment.

In [ ]:
# ▶ Run this cell first. It defines check() and expect(), which test your answers,
#   and DATA, the path to the course's shared data folder.
import copy, math
from pathlib import Path

DATA = next((p / "data" for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "README.md").exists()), Path("data"))

def _same(a, b):
    if hasattr(b, "shape") and hasattr(b, "dtype"):          # numpy arrays
        import numpy as np
        try:
            return np.shape(a) == np.shape(b) and np.allclose(a, b, rtol=1e-5, atol=1e-8)
        except TypeError:
            return False
    if isinstance(b, float) and isinstance(a, (int, float)) and not isinstance(a, bool):
        return math.isclose(a, b, rel_tol=1e-6, abs_tol=1e-9)
    if isinstance(b, (list, tuple)):
        return type(a) is type(b) and len(a) == len(b) and all(_same(x, y) for x, y in zip(a, b))
    if isinstance(b, dict):
        return isinstance(a, dict) and a.keys() == b.keys() and all(_same(a[k], b[k]) for k in b)
    return a == b

def check(fn, cases):
    """Run fn on each (args, expected) pair and report. args is a tuple of arguments."""
    ok = 0
    for args, expected in cases:
        if not isinstance(args, tuple):
            args = (args,)
        shown = ", ".join(repr(a) for a in args)
        if len(shown) > 80:
            shown = shown[:77] + "..."
        try:
            got = fn(*copy.deepcopy(args))
        except Exception as e:
            print(f"  ✗ {fn.__name__}({shown}) raised {type(e).__name__}: {e}")
            continue
        if _same(got, expected):
            ok += 1
        else:
            print(f"  ✗ {fn.__name__}({shown}) returned {got!r}, expected {expected!r}")
    status = "✅ all passed" if ok == len(cases) else "keep going"
    print(f"{fn.__name__}: {ok}/{len(cases)} passed — {status}")

def expect(description, condition):
    """For checks that aren't a simple function call: expect('mean is ~0', abs(m) < 1e-6)."""
    try:
        passed = bool(condition() if callable(condition) else condition)
    except Exception as e:
        print(f"  ✗ {description} — raised {type(e).__name__}: {e}")
        return
    print(("  ✅ " if passed else "  ✗ ") + description + ("" if passed else " — keep going"))


In [ ]:
def largest_gap(amounts):
    biggest = 0
    for i in range(1, len(amounts)):
        gap = abs(amounts[i] - amounts[i - 1])
        if gap > biggest:
            biggest = gap
    return biggest

largest_gap([3, 11, 4, 4, 12])

## Problem 2 — find the bug with the debugger

`count_above(values, threshold)` should count how many values are **strictly greater** than `threshold`. It returns wrong answers. Don't stare at it — set a breakpoint, step through the first example, and watch `count`. Then fix it.

In [ ]:
def count_above(values, threshold):
    count = 0
    for v in values:
        if v > threshold:
            count = 1
    return count

In [ ]:
check(count_above, [
    (([5, 1, 9, 7], 4), 3),
    (([1, 2], 5), 0),
    (([5, 5, 6], 5), 1),
])

## Problem 3 — step into

`net_pay` calls `tax`. Put a breakpoint on the `return` line of `net_pay`, debug the cell, and use **Step Into** (`F11`) to go inside `tax`. Find and fix the bug. The rule is: *no tax on the first 100,000; 24% on the rest* (so a salary of 200,000 gives a net pay of 176,000).

In [1]:
def tax(salary):
    taxable = salary - 100_000
    return taxable * 0.24

def net_pay(salary):
    return salary - tax(salary)

In [2]:
check(net_pay, [((200_000,), 176_000.0), ((80_000,), 80_000), ((100_000,), 100_000)])

NameError: name 'check' is not defined